# Strategy Tester (No TBM / CPCV / HMM / XGBoost)

Goal:
1. Compare multiple raw strategy families under one unified backtester.
2. Include realistic friction checks (spread cap, slippage).
3. Rank by robustness (plateau stability) and profit-per-trade.
4. Export top candidates for later wiring into GoldRegimeX_Explorer.

In [1]:
# === Shared library imports (feature engineering + session helpers) ===
# The strategy backtester reuses the shared feature-engineering and session
# helpers in pipeline_verification_bundle/shared so they stay identical across
# notebooks. This cell only locates the bundle and imports those helpers.
import os, sys
from pathlib import Path

_MARKER = "pipeline_verification_bundle"

def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for cand in (start, *start.parents):
        if (cand / _MARKER / "shared" / "features.py").exists():
            return cand
    for cand in (start, *start.parents):
        if (cand / _MARKER).is_dir():
            return cand
    raise FileNotFoundError("Could not locate %s from %s" % (_MARKER, start))

_search = []
for _v in ("__vsc_ipynb_file__", "__session__"):
    _val = globals().get(_v)
    if isinstance(_val, str) and _val:
        _search.append(Path(_val).parent)
_search.append(Path.cwd())
_repo_root = None
for _s in _search:
    try:
        _repo_root = _find_repo_root(_s); break
    except FileNotFoundError:
        continue
if _repo_root is None:
    raise FileNotFoundError("pipeline_verification_bundle not found from: %s" % _search)
os.chdir(_repo_root)
for _p in (_repo_root, _repo_root / _MARKER, _repo_root / _MARKER / "shared"):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from shared.features import build_features, ema, true_range, atr, rsi, adx
from shared.session_filter import session_col_from_value

print("[shared] repo_root =", _repo_root)


[shared] repo_root = C:\GoldRegime_X


In [2]:
# Imports + Config
import os
import math
import json
import time
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from joblib import Parallel, delayed
    JOBLIB_OK = True
except Exception:
    JOBLIB_OK = False

np.random.seed(42)

# -----------------------------
# Runtime controls
# -----------------------------
N_JOBS = 6  # lower to reduce memory pressure under loky
QUICK_MODE = False
RESEARCH_YEARS = 10 # enforced always, regardless of QUICK_MODE

# -----------------------------
# Data paths
# -----------------------------
M5_PATH = Path("data/raw/XAU_5m_data.csv")
M15_PATH = Path("data/raw/XAU_15m_data.csv")

# -----------------------------
# Core assumptions
# -----------------------------
TIMEFRAMES = ["M15", "M5"]
INITIAL_BALANCE_CENTS = 1500.0
PIP_SIZE_PRICE = 0.10
PIP_VALUE_CENTS_PER_1LOT = 100.0
SLIPPAGE_PIPS = 0.30
SPREAD_CAP_POINTS = 40.0
COMMISSION_CENTS_PER_TRADE = 0.0

# Split position support
POSITION_A = 0.05
POSITION_B = 0.05

# M5 Grids (Tighter targets, wider stops to survive noise and high friction)
M5_ADX_GRID = [20, 25]
M5_PULLBACK_RSI_GRID = [30, 35]
M5_CONFIRMATION_GRID = [1, 2]
M5_ATR_STOP_GRID = [1.0, 1.5, 2.0, 2.5, 3.0]  # Extended upward to allow M5 breathing room
M5_LEG_A_TARGET_GRID = [0.8, 1.0]
M5_ENTRY_TARGET_GRID = [1.0, 1.5]

# Exit modes
EXIT_MODELS = [
    "fixed_tp",
    "mr_exit",
    "fixed_tp_plus_mr",
    "partial_tp_plus_mr",
    "partial_tp_mr_time_stop",
]

# ATR multiple grids
LEG_A_ATR_TARGET_GRID = [1.0, 1.5, 2.0]  # Leg A target (ATR multiple)
ENTRY_ATR_TARGET_GRID = [2.0, 2.5, 3.0]  # Leg B (fixed TP) ATR multiple
ATR_TARGET_GRID = ENTRY_ATR_TARGET_GRID  # keep existing name used by strategies

# Leg C constants (Scale-in removed from grid search to save compute)
LEG_C_ATR_TARGET = 0.5
LEG_C_ATR_STOP = 0.5

TIME_STOP_GRID_BY_TF = {
    "M15": [120, 180, 240],
    "M5": [30, 60, 90],
}
TRAIL_MULT_GRID = [1.5, 2.0, 2.5]

# Session filter options
SESSION_FILTER_VALUES = [None, "London", "NY", "London_NY"]

# -----------------------------
# Strategy A grids (Loosened heavily to restore M15 trade frequency)
# -----------------------------
# Lowered ADX so we don't need a massive macro trend to trigger
ADX_GRID = [15.0, 18.0, 20.0, 25.0] 
# Loosened RSI heavily. On a 15-minute chart, trends rarely retrace all the way to 25/30 RSI. 
# Allowing 35-45 RSI captures shallower, highly valid trend pullbacks.
PULLBACK_RSI_GRID = [25, 30, 35.0, 40.0, 45.0] 
CONFIRMATION_GRID = [1, 2]  # Get in faster, less lag
ATR_STOP_GRID = [0.8, 1.0, 1.5, 2.0, 2.5, 3.0]  # Loosened to allow more breathing room for M15 noise

# -----------------------------
# Strategy B grids (Loosened for Frequency)
# -----------------------------
ATR_EXPANSION_GRID = [1.1, 1.25, 1.4]  # 1.5+ is too rare; 1.1+ catches standard volatility cycles
BREAKOUT_LOOKBACK_GRID = [10, 20, 30]
BREAKOUT_BUFFER_GRID = [0.0, 0.25]

if QUICK_MODE:
    # Stronger quick cut to keep runtime practical
    ADX_GRID = ADX_GRID[:2]
    PULLBACK_RSI_GRID = PULLBACK_RSI_GRID[:2]
    CONFIRMATION_GRID = CONFIRMATION_GRID[:2]
    ATR_STOP_GRID = ATR_STOP_GRID[:1]
    LEG_A_ATR_TARGET_GRID = LEG_A_ATR_TARGET_GRID[:2]
    ENTRY_ATR_TARGET_GRID = ENTRY_ATR_TARGET_GRID[:2]
    ATR_TARGET_GRID = ENTRY_ATR_TARGET_GRID

    ATR_EXPANSION_GRID = ATR_EXPANSION_GRID[:3]
    BREAKOUT_LOOKBACK_GRID = BREAKOUT_LOOKBACK_GRID[:2]
    BREAKOUT_BUFFER_GRID = BREAKOUT_BUFFER_GRID[:2]

    SESSION_FILTER_VALUES = [None, "London_NY"]
    EXIT_MODELS = ["fixed_tp", "mr_exit", "fixed_tp_plus_mr"]

    TIME_STOP_GRID_BY_TF["M15"] = TIME_STOP_GRID_BY_TF["M15"][:2]
    TIME_STOP_GRID_BY_TF["M5"] = TIME_STOP_GRID_BY_TF["M5"][:2]
    TRAIL_MULT_GRID = TRAIL_MULT_GRID[:2]

print("JOBLIB_OK:", JOBLIB_OK)
print("N_JOBS:", N_JOBS)
print("QUICK_MODE:", QUICK_MODE)
print("RESEARCH_YEARS (enforced):", RESEARCH_YEARS)

JOBLIB_OK: True
N_JOBS: 6
QUICK_MODE: False
RESEARCH_YEARS (enforced): 10


In [3]:
# Optional exploratory speed cap (applies to QUICK_MODE True/False)
ENABLE_ENTRY_CAP = False

# Hard cap per (timeframe, strategy, exit_model) on ENTRY parameter combos.
# This is the biggest speed lever for QUICK_MODE=False.
if QUICK_MODE:
    ENTRY_CAP_BY_TF = {"M15": 80, "M5": 40}
else:
    ENTRY_CAP_BY_TF = {"M15": 200, "M5": 80}

# Seed for deterministic subset selection
ENTRY_CAP_SEED_BASE = 42

print("ENABLE_ENTRY_CAP:", ENABLE_ENTRY_CAP)
print("ENTRY_CAP_BY_TF:", ENTRY_CAP_BY_TF)
print("ENTRY_CAP_SEED_BASE:", ENTRY_CAP_SEED_BASE)

ENABLE_ENTRY_CAP: False
ENTRY_CAP_BY_TF: {'M15': 200, 'M5': 80}
ENTRY_CAP_SEED_BASE: 42


In [4]:
# Data loading, strict 5-year reduction, indicators, and rule-based regimes

def resolve_path(path: Path) -> Path:
    if path.exists():
        return path
    alt = Path.cwd().parent / path
    if alt.exists():
        return alt
    raise FileNotFoundError(f"Path not found: {path}")

def _normalize_ohlc(df: pd.DataFrame) -> pd.DataFrame:
    cols = {c.lower(): c for c in df.columns}
    rename = {}
    for key in ["open", "high", "low", "close", "volume", "spread"]:
        if key in cols:
            rename[cols[key]] = key
    out = df.rename(columns=rename)
    missing = [c for c in ["open", "high", "low", "close"] if c not in out.columns]
    if missing:
        raise ValueError(f"Missing OHLC columns: {missing}")
    return out

def read_xau_raw(path: Path) -> pd.DataFrame:
    path = resolve_path(path)
    df = pd.read_csv(path, sep=";")
    if "Date" not in df.columns:
        raise ValueError(f"Date column missing in {path}")
    df["Date"] = pd.to_datetime(df["Date"], format="%Y.%m.%d %H:%M", errors="coerce")
    df = df.dropna(subset=["Date"]).set_index("Date").sort_index()
    df = _normalize_ohlc(df)
    if "spread" not in df.columns:
        df["spread"] = SPREAD_CAP_POINTS
    return df

def load_recent_years(df: pd.DataFrame, years: int) -> pd.DataFrame:
    last_date = df.index.max()
    start_date = last_date - pd.DateOffset(years=int(years))
    return df.loc[df.index >= start_date].copy()

def enforce_recent_window(df_full: pd.DataFrame, df_trim: pd.DataFrame, years: int, label: str):
    last_date = df_full.index.max()
    target_start = last_date - pd.DateOffset(years=int(years))
    observed_start = df_trim.index.min()
    if observed_start < target_start:
        raise RuntimeError(f"{label} window enforcement failed: observed_start={observed_start}, target_start={target_start}")
    print(f"{label} enforced window: {observed_start} -> {last_date}")

def ema(series: pd.Series, period: int) -> pd.Series:
    return series.ewm(span=int(period), adjust=False, min_periods=int(period)).mean()

def true_range(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev_close = close.shift(1)
    return pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)

def atr(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    tr = true_range(high, low, close)
    return tr.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()

def rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return (100.0 - (100.0 / (1.0 + rs))).fillna(50.0)

def adx(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    up_move = high.diff()
    down_move = -low.diff()
    plus_dm = up_move.where((up_move > down_move) & (up_move > 0.0), 0.0)
    minus_dm = down_move.where((down_move > up_move) & (down_move > 0.0), 0.0)
    atr_v = atr(high, low, close, period=period).replace(0.0, np.nan)
    plus_di = 100.0 * plus_dm.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean() / atr_v
    minus_di = 100.0 * minus_dm.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean() / atr_v
    dx = 100.0 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0.0, np.nan)
    return dx.ewm(alpha=1.0 / period, adjust=False, min_periods=period).mean().fillna(0.0)

def add_session_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    hour = out.index.hour
    london = (hour >= 7) & (hour < 16)
    ny = (hour >= 13) & (hour < 21)
    overlap = (hour >= 13) & (hour < 16)
    out["session"] = np.where(overlap, "OVERLAP", np.where(london, "LONDON", np.where(ny, "NEW_YORK", "ASIA")))
    out["session_mask_none"] = True
    out["session_mask_london"] = london
    out["session_mask_ny"] = ny
    out["session_mask_london_ny"] = london | ny
    return out

def build_features(exec_tf: str, m5_df: pd.DataFrame, m15_df: pd.DataFrame) -> pd.DataFrame:
    exec_tf = exec_tf.upper()
    if exec_tf not in ("M5", "M15"):
        raise ValueError(f"Unsupported timeframe: {exec_tf}")

    exec_df = m5_df.copy() if exec_tf == "M5" else m15_df.copy()
    trend_df = m15_df.copy()

    exec_df["rsi5"] = rsi(exec_df["close"], period=5)
    exec_df["atr14"] = atr(exec_df["high"], exec_df["low"], exec_df["close"], period=14)
    exec_df["atr100"] = atr(exec_df["high"], exec_df["low"], exec_df["close"], period=100)
    exec_df["atr_expansion"] = exec_df["atr14"] / exec_df["atr100"].replace(0.0, np.nan)

    for lb in sorted(set(BREAKOUT_LOOKBACK_GRID)):
        exec_df[f"roll_high_{lb}"] = exec_df["high"].rolling(lb, min_periods=lb).max().shift(1)
        exec_df[f"roll_low_{lb}"] = exec_df["low"].rolling(lb, min_periods=lb).min().shift(1)

    trend_df["m15_ema50"] = ema(trend_df["close"], period=50)
    trend_df["m15_ema200"] = ema(trend_df["close"], period=200)
    trend_df["m15_adx14"] = adx(trend_df["high"], trend_df["low"], trend_df["close"], period=14)

    if exec_tf == "M5":
        ex = exec_df.reset_index().rename(columns={exec_df.index.name or "index": "time"})
        tr = trend_df.reset_index().rename(columns={trend_df.index.name or "index": "time"})
        merged = pd.merge_asof(
            ex.sort_values("time"),
            tr[["time", "m15_ema50", "m15_ema200", "m15_adx14"]].sort_values("time"),
            on="time",
            direction="backward",
        ).set_index("time")
    else:
        merged = exec_df.copy()
        merged["m15_ema50"] = trend_df["m15_ema50"].reindex(merged.index)
        merged["m15_ema200"] = trend_df["m15_ema200"].reindex(merged.index)
        merged["m15_adx14"] = trend_df["m15_adx14"].reindex(merged.index)

    merged = add_session_features(merged)
    merged["spread"] = merged["spread"].fillna(SPREAD_CAP_POINTS)

        # Rule-based regime classification
    is_trend = (merged["m15_adx14"] > 15.0) & (merged["atr_expansion"] < 1.3)
    is_shock = merged["atr_expansion"] >= 1.3

    merged["regime_str"] = np.where(is_shock, "SHOCK", np.where(is_trend, "TREND", "MR"))
    merged["regime_code"] = np.where(is_shock, 2, np.where(is_trend, 1, 3)).astype(np.int32)
    
    required = [
        "open", "high", "low", "close", "spread",
        "rsi5", "atr14", "atr100", "atr_expansion",
        "m15_ema50", "m15_ema200", "m15_adx14",
        "session", "regime_str", "regime_code",
        "session_mask_none", "session_mask_london", "session_mask_ny", "session_mask_london_ny",
    ]
    merged = merged.dropna(subset=[c for c in required if c in merged.columns]).copy()
    return merged

m5_raw_full = read_xau_raw(M5_PATH)
m15_raw_full = read_xau_raw(M15_PATH)

m5_raw = load_recent_years(m5_raw_full, years=RESEARCH_YEARS)
m15_raw = load_recent_years(m15_raw_full, years=RESEARCH_YEARS)

enforce_recent_window(m5_raw_full, m5_raw, years=RESEARCH_YEARS, label="M5")
enforce_recent_window(m15_raw_full, m15_raw, years=RESEARCH_YEARS, label="M15")

FEATURES_BY_TF = {
    "M5": build_features("M5", m5_raw, m15_raw),
    "M15": build_features("M15", m5_raw, m15_raw),
}

print("M5 full rows:", len(m5_raw_full), "| recent rows:", len(m5_raw))
print("M15 full rows:", len(m15_raw_full), "| recent rows:", len(m15_raw))
for tf in TIMEFRAMES:
    print(tf, "feature rows:", len(FEATURES_BY_TF[tf]))
display(FEATURES_BY_TF["M5"].head(3))

M5 enforced window: 2016-05-30 01:05:00 -> 2026-05-29 23:50:00
M15 enforced window: 2016-05-30 01:00:00 -> 2026-05-29 23:30:00
M5 full rows: 1468386 | recent rows: 697853
M15 full rows: 502563 | recent rows: 233709
M15 feature rows: 233510
M5 feature rows: 697261


,open,high,low,close,volume,spread,rsi5,atr14,atr100,atr_expansion,...,m15_ema50,m15_ema200,m15_adx14,session,session_mask_none,session_mask_london,session_mask_ny,session_mask_london_ny,regime_str,regime_code
time,,,,,,,,,,,,,,,,,,,,,
2016-06-01 10:45:00,1216.12,1216.43,1215.44,1216.18,467,40.0,41.274045,0.926701,0.775095,1.195596,...,1215.555016,1213.039968,17.782744,LONDON,True,True,False,True,TREND,1
2016-06-01 10:50:00,1216.16,1216.40,1215.75,1216.18,298,40.0,41.274045,0.906937,0.773844,1.171989,...,1215.555016,1213.039968,17.782744,LONDON,True,True,False,True,TREND,1
2016-06-01 10:55:00,1216.15,1216.71,1216.15,1216.37,326,40.0,47.823850,0.882156,0.771706,1.143124,...,1215.555016,1213.039968,17.782744,LONDON,True,True,False,True,TREND,1


In [5]:
# Legacy cell disabled intentionally.
# Data loading and feature building are now centralized in the previous cell
# and strictly enforced to RESEARCH_YEARS only.

print("Legacy research-frame cell disabled. Using FEATURES_BY_TF from strict recent-window pipeline.")

Legacy research-frame cell disabled. Using FEATURES_BY_TF from strict recent-window pipeline.


In [6]:
# Cell 5: Strategy Definitions & Regime-Based Signal Router (With Macro Filter)
class BaseStrategy:
    name = "base"
    param_grid = {}
    param_cols = []

    def iter_param_dicts(self):
        keys = list(self.param_cols)
        vals = [self.param_grid[k] for k in keys]
        for combo in itertools.product(*vals):
            yield dict(zip(keys, combo))

    def generate_signals(self, df: pd.DataFrame, params: dict) -> pd.Series:
        raise NotImplementedError

def _legacy_session_col_from_value(session_filter):
    if session_filter is None:
        return "session_mask_none"
    s = str(session_filter).lower()
    if s == "london":
        return "session_mask_london"
    if s == "ny":
        return "session_mask_ny"
    if s == "london_ny":
        return "session_mask_london_ny"
    raise ValueError(f"Unsupported session_filter: {session_filter}")

class TrendPullbackStrategy(BaseStrategy):
    name = "trend_pullback"
    param_grid = {
        "adx_threshold": ADX_GRID,
        "pullback_rsi": PULLBACK_RSI_GRID,
        "confirmation_bars": CONFIRMATION_GRID,
        "atr_stop": ATR_STOP_GRID,
        "atr_target": ATR_TARGET_GRID,
        "session_filter": SESSION_FILTER_VALUES,
    }
    param_cols = list(param_grid.keys())

    def generate_signals(self, df: pd.DataFrame, params: dict) -> pd.Series:
        adx_threshold = float(params["adx_threshold"])
        pullback_rsi = float(params["pullback_rsi"])
        confirmation_bars = int(params["confirmation_bars"])
        session_col = session_col_from_value(params["session_filter"])

        trend_up_raw = (df["m15_ema50"] > df["m15_ema200"]) & (df["m15_adx14"] > adx_threshold)
        trend_dn_raw = (df["m15_ema50"] < df["m15_ema200"]) & (df["m15_adx14"] > adx_threshold)

        if confirmation_bars > 1:
            trend_up = trend_up_raw.rolling(confirmation_bars, min_periods=confirmation_bars).sum().eq(confirmation_bars)
            trend_dn = trend_dn_raw.rolling(confirmation_bars, min_periods=confirmation_bars).sum().eq(confirmation_bars)
        else:
            trend_up, trend_dn = trend_up_raw, trend_dn_raw

        long_cond = trend_up & (df["rsi5"] < pullback_rsi) & df[session_col].astype(bool)
        short_cond = trend_dn & (df["rsi5"] > (100.0 - pullback_rsi)) & df[session_col].astype(bool)

        sig = pd.Series(0, index=df.index, dtype=np.int8)
        sig.loc[long_cond.fillna(False)] = 1
        sig.loc[short_cond.fillna(False)] = -1
        return sig

class VolatilityExpansionStrategy(BaseStrategy):
    name = "volatility_expansion"
    param_grid = {
        "atr_expansion_threshold": ATR_EXPANSION_GRID,
        "breakout_lookback": BREAKOUT_LOOKBACK_GRID,
        "breakout_buffer": BREAKOUT_BUFFER_GRID,
        "atr_stop": ATR_STOP_GRID,
        "atr_target": ATR_TARGET_GRID,
        "session_filter": SESSION_FILTER_VALUES,
    }
    param_cols = list(param_grid.keys())

    def generate_signals(self, df: pd.DataFrame, params: dict) -> pd.Series:
        thr = float(params["atr_expansion_threshold"])
        lb = int(params["breakout_lookback"])
        buf_mult = float(params["breakout_buffer"])
        session_col = session_col_from_value(params["session_filter"])

        high_col = f"roll_high_{lb}"
        low_col = f"roll_low_{lb}"
        breakout_buffer = buf_mult * df["atr14"]
        is_expansion = df["atr_expansion"] > thr

        # MACRO TREND FILTER: Ensures entries align with higher timeframe momentum
        macro_up = df["m15_ema50"] > df["m15_ema200"]
        macro_dn = df["m15_ema50"] < df["m15_ema200"]

        long_cond = is_expansion & (df["close"] > (df[high_col] + breakout_buffer)) & macro_up & df[session_col].astype(bool)
        short_cond = is_expansion & (df["close"] < (df[low_col] - breakout_buffer)) & macro_dn & df[session_col].astype(bool)

        sig = pd.Series(0, index=df.index, dtype=np.int8)
        sig.loc[long_cond.fillna(False)] = 1
        sig.loc[short_cond.fillna(False)] = -1
        return sig

STRATEGIES = {
    "trend_pullback": TrendPullbackStrategy(),
    "volatility_expansion": VolatilityExpansionStrategy(),
}

def generate_routed_signals(df: pd.DataFrame, params: dict, strategy_name: str) -> pd.Series:
    raw_signals = STRATEGIES[strategy_name].generate_signals(df, params)
    routed = pd.Series(0, index=df.index, dtype=np.int8)
    if strategy_name == "trend_pullback":
        mask = df["regime_code"] == 1
        routed.loc[mask] = raw_signals.loc[mask]
    elif strategy_name == "volatility_expansion":
        mask = df["regime_code"] == 2
        routed.loc[mask] = raw_signals.loc[mask]
    return routed

In [7]:
# Cell 6: Numba Backtest Engine with Early Eject, Pyramiding (Scale-In), and Asymmetric Guard

from numba import njit
import numpy as np
import pandas as pd
import math

def compute_metrics(trades_df: pd.DataFrame, equity_curve: list[float], initial_balance: float, ending_balance: float) -> dict:
    if trades_df.empty:
        return {
            "profit_factor": 0.0, "sharpe": 0.0, "sortino": 0.0, "calmar": 0.0, 
            "max_drawdown": 0.0, "expectancy": 0.0, "win_rate": 0.0, "trade_count": 0, 
            "net_profit": float(ending_balance - initial_balance), "profit_per_trade": 0.0, "net_return_pct": 0.0
        }

    pnl = trades_df["pnl_cents"].to_numpy(dtype=float)
    trade_count = int(len(pnl))
    net_profit = float(np.sum(pnl))
    gross_profit = float(np.sum(pnl[pnl > 0]))
    gross_loss = float(-np.sum(pnl[pnl < 0]))
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else (10.0 if gross_profit > 0 else 0.0)
    win_rate = float(np.mean(pnl > 0))
    expectancy = float(np.mean(pnl))
    profit_per_trade = float(net_profit / max(trade_count, 1))

    r = pnl / max(initial_balance, 1e-9)
    r_std = float(np.std(r, ddof=0))
    sharpe = float((np.mean(r) / r_std) * math.sqrt(len(r))) if r_std > 0 else 0.0

    downside = r[r < 0]
    d_std = float(np.std(downside, ddof=0)) if len(downside) > 0 else 0.0
    sortino = float((np.mean(r) / d_std) * math.sqrt(len(r))) if d_std > 0 else 0.0

    eq = np.array(equity_curve, dtype=float)
    
    # RISK MANAGEMENT: Drawdown calculations based on an average of multiple positions
    window = min(3, len(eq))
    avg_eq = pd.Series(eq).rolling(window=window, min_periods=1).mean().to_numpy()
    peaks = np.maximum.accumulate(avg_eq)
    dd = peaks - avg_eq
    max_dd_pct = float(np.max(dd / np.maximum(peaks, 1e-9)) * 100.0) if len(dd) > 0 else 0.0
    
    net_return_pct = float((ending_balance / initial_balance - 1.0) * 100.0)
    calmar = float(net_return_pct / max_dd_pct) if max_dd_pct > 0 else 0.0

    return {
        "profit_factor": profit_factor, "sharpe": sharpe, "sortino": sortino, "calmar": calmar, 
        "max_drawdown": max_dd_pct, "expectancy": expectancy, "win_rate": win_rate, "trade_count": trade_count, 
        "net_profit": net_profit, "profit_per_trade": profit_per_trade, "net_return_pct": net_return_pct
    }

@njit(cache=True)
def _run_backtest_numba(
    sig, high, low, close, spread, atr14, regime_code, time_minutes, 
    entry_atr_stop, entry_atr_target, leg_a_atr_target, exit_mode_code, 
    time_stop_minutes, trail_mult, initial_balance, pip_size_price, 
    pip_value_per_1lot, slippage_pips, spread_cap_points, commission_cents, is_m5
):
    n = sig.shape[0]
    max_trades = n * 3

    trade_entry_idx = np.empty(max_trades, dtype=np.int64)
    trade_exit_idx = np.empty(max_trades, dtype=np.int64)
    trade_leg = np.empty(max_trades, dtype=np.int8)
    trade_side = np.empty(max_trades, dtype=np.int8)
    trade_entry_px = np.empty(max_trades, dtype=np.float64)
    trade_exit_px = np.empty(max_trades, dtype=np.float64)
    trade_move_pips = np.empty(max_trades, dtype=np.float64)
    trade_pnl_cents = np.empty(max_trades, dtype=np.float64)
    trade_reason = np.empty(max_trades, dtype=np.int8)
    trade_entry_regime = np.empty(max_trades, dtype=np.int32)

    eq = np.empty(max_trades + 1, dtype=np.float64)
    eq[0] = initial_balance
    eq_count = 1

    legs = np.zeros((3, 7), dtype=np.float64)
    trade_count = 0
    balance = initial_balance
    active_trade_regime = 0
    leg_a_profit_hit = 0
    leg_b_profit_hit = 0

    enable_fixed_tp = exit_mode_code in (0, 2)
    enable_mr = exit_mode_code in (1, 2, 3, 4)
    enable_time_stop = exit_mode_code == 4
    enable_trail = exit_mode_code == 4

    for i in range(n):
        s = int(sig[i])
        current_regime = int(regime_code[i])

        is_flat = (legs[0, 0] == 0.0 and legs[1, 0] == 0.0 and legs[2, 0] == 0.0)
        leg_a_closed_b_open = (legs[0, 0] == 0.0 and legs[1, 0] == 1.0 and leg_a_profit_hit == 1)
        leg_b_closed_a_open = (legs[1, 0] == 0.0 and legs[0, 0] == 1.0 and leg_b_profit_hit == 1)
        can_scale_in = (legs[2, 0] == 0.0 and (leg_a_closed_b_open or leg_b_closed_a_open))

        # ENTRY & SCALE-IN LOGIC
        if (is_flat or can_scale_in) and s != 0:
            sp_points = spread[i]
            atr_now = atr14[i]
            
            effective_spread = (sp_points / 10.0) * pip_size_price
            if np.isfinite(atr_now) and atr_now > 0.0 and effective_spread <= (0.8 * atr_now):
                side = 1 if s > 0 else -1

                if can_scale_in:
                    runner_idx = 1 if leg_a_closed_b_open else 0
                    if side != int(legs[runner_idx, 2]):
                        continue 

                spread_price = effective_spread
                slippage_price = slippage_pips * pip_size_price
                close_px = close[i]

                # RISK MANAGEMENT: Dynamic Asymmetric Guard Factor
                if is_m5 == 1:
                    guard_factor = 0.85 if side == 1 else 0.75
                else:
                    guard_factor = 0.65

                if is_flat:
                    leg_a_profit_hit = 0
                    leg_b_profit_hit = 0
                    
                    intended_stop_dist = entry_atr_stop * atr_now
                    actual_stop_dist = intended_stop_dist * guard_factor
                    tp_dist = entry_atr_target * atr_now

                    if side == 1:
                        entry_px = close_px + spread_price + slippage_price
                        stop_px = entry_px - actual_stop_dist
                        fixed_tp_px = entry_px + tp_dist
                        leg_a_tp_px = entry_px + (leg_a_atr_target * atr_now) if leg_a_atr_target > 0.0 else np.nan
                    else:
                        entry_px = close_px - slippage_price
                        stop_px = entry_px + actual_stop_dist + spread_price
                        fixed_tp_px = entry_px - tp_dist
                        leg_a_tp_px = entry_px - (leg_a_atr_target * atr_now) if leg_a_atr_target > 0.0 else np.nan

                    active_trade_regime = current_regime
                    legs[0, 0], legs[0, 1], legs[0, 2], legs[0, 3], legs[0, 4], legs[0, 5], legs[0, 6] = 1.0, POSITION_A, side, entry_px, stop_px, leg_a_tp_px, i
                    legs[1, 0], legs[1, 1], legs[1, 2], legs[1, 3], legs[1, 4], legs[1, 5], legs[1, 6] = 1.0, POSITION_B, side, entry_px, stop_px, fixed_tp_px if enable_fixed_tp else np.nan, i

                elif can_scale_in:
                    leg_c_lot = POSITION_A if leg_a_closed_b_open else POSITION_B
                    
                    tighter_stop = 0.5 * atr_now
                    actual_stop_dist = tighter_stop * guard_factor
                    tighter_tp = 0.5 * atr_now

                    if side == 1:
                        entry_px = close_px + spread_price + slippage_price
                        stop_px = entry_px - actual_stop_dist
                        fixed_tp_px = entry_px + tighter_tp
                    else:
                        entry_px = close_px - slippage_price
                        stop_px = entry_px + actual_stop_dist + spread_price
                        fixed_tp_px = entry_px - tighter_tp

                    legs[2, 0], legs[2, 1], legs[2, 2], legs[2, 3], legs[2, 4], legs[2, 5], legs[2, 6] = 1.0, leg_c_lot, side, entry_px, stop_px, fixed_tp_px, i

        bar_high, bar_low, bar_close, atr_now = high[i], low[i], close[i], atr14[i]

        # EXIT PROCESSING
        for leg_idx in range(3):
            if legs[leg_idx, 0] == 0.0:
                continue

            leg_side = int(legs[leg_idx, 2])
            leg_entry = legs[leg_idx, 3]
            leg_stop = legs[leg_idx, 4]
            leg_tp = legs[leg_idx, 5]
            leg_lot = legs[leg_idx, 1]
            leg_entry_i = int(legs[leg_idx, 6])

            if enable_trail and leg_idx == 1 and np.isfinite(atr_now) and atr_now > 0.0:
                dist = trail_mult * atr_now
                if leg_side == 1:
                    if bar_close - dist > leg_stop:
                        leg_stop = bar_close - dist
                else:
                    if bar_close + dist < leg_stop:
                        leg_stop = bar_close + dist
                legs[leg_idx, 4] = leg_stop

            # 1. Stop Loss Hit
            stop_hit = (bar_low <= leg_stop) if leg_side == 1 else (bar_high >= leg_stop)
            if stop_hit:
                exit_px = leg_stop
                move_pips = ((exit_px - leg_entry) / pip_size_price) * leg_side
                pnl_cents = move_pips * (leg_lot * pip_value_per_1lot) - commission_cents
                trade_entry_idx[trade_count] = leg_entry_i
                trade_exit_idx[trade_count] = i
                trade_leg[trade_count] = leg_idx
                trade_side[trade_count] = leg_side
                trade_entry_px[trade_count] = leg_entry
                trade_exit_px[trade_count] = exit_px
                trade_move_pips[trade_count] = move_pips
                trade_pnl_cents[trade_count] = pnl_cents
                trade_reason[trade_count] = 1
                trade_entry_regime[trade_count] = active_trade_regime
                trade_count += 1
                balance += pnl_cents
                eq[eq_count] = balance
                eq_count += 1
                legs[leg_idx, 0] = 0.0
                if leg_idx == 1: leg_a_profit_hit = 0
                continue

            # 2. Leg 0 (A) Profit Target
            if leg_idx == 0 and np.isfinite(leg_tp):
                tp_hit = (bar_high >= leg_tp) if leg_side == 1 else (bar_low <= leg_tp)
                if tp_hit:
                    exit_px = leg_tp
                    move_pips = ((exit_px - leg_entry) / pip_size_price) * leg_side
                    pnl_cents = move_pips * (leg_lot * pip_value_per_1lot) - commission_cents
                    trade_entry_idx[trade_count] = leg_entry_i
                    trade_exit_idx[trade_count] = i
                    trade_leg[trade_count] = leg_idx
                    trade_side[trade_count] = leg_side
                    trade_entry_px[trade_count] = leg_entry
                    trade_exit_px[trade_count] = exit_px
                    trade_move_pips[trade_count] = move_pips
                    trade_pnl_cents[trade_count] = pnl_cents
                    trade_reason[trade_count] = 2
                    trade_entry_regime[trade_count] = active_trade_regime
                    trade_count += 1
                    balance += pnl_cents
                    eq[eq_count] = balance
                    eq_count += 1
                    legs[leg_idx, 0] = 0.0
                    leg_a_profit_hit = 1
                    continue

            # 3. Leg B or Leg C Fixed TP
            if leg_idx in (1, 2) and enable_fixed_tp and np.isfinite(leg_tp):
                tp_hit = (bar_high >= leg_tp) if leg_side == 1 else (bar_low <= leg_tp)
                if tp_hit:
                    exit_px = leg_tp
                    move_pips = ((exit_px - leg_entry) / pip_size_price) * leg_side
                    pnl_cents = move_pips * (leg_lot * pip_value_per_1lot) - commission_cents
                    trade_entry_idx[trade_count] = leg_entry_i
                    trade_exit_idx[trade_count] = i
                    trade_leg[trade_count] = leg_idx
                    trade_side[trade_count] = leg_side
                    trade_entry_px[trade_count] = leg_entry
                    trade_exit_px[trade_count] = exit_px
                    trade_move_pips[trade_count] = move_pips
                    trade_pnl_cents[trade_count] = pnl_cents
                    trade_reason[trade_count] = 3
                    trade_entry_regime[trade_count] = active_trade_regime
                    trade_count += 1
                    balance += pnl_cents
                    eq[eq_count] = balance
                    eq_count += 1
                    legs[leg_idx, 0] = 0.0
                    if leg_idx == 1:
                        leg_b_profit_hit = 1
                    continue

            # 4. Structural MR Exit (Clears all active legs)
            if leg_idx == 1 and enable_mr and current_regime == 3:
                for l_id in range(3):
                    if legs[l_id, 0] == 1.0:
                        m_pips = ((bar_close - legs[l_id, 3]) / pip_size_price) * int(legs[l_id, 2])
                        p_cents = m_pips * (legs[l_id, 1] * pip_value_per_1lot) - commission_cents
                        trade_entry_idx[trade_count] = int(legs[l_id, 6])
                        trade_exit_idx[trade_count] = i
                        trade_leg[trade_count] = l_id
                        trade_side[trade_count] = int(legs[l_id, 2])
                        trade_entry_px[trade_count] = legs[l_id, 3]
                        trade_exit_px[trade_count] = bar_close
                        trade_move_pips[trade_count] = m_pips
                        trade_pnl_cents[trade_count] = p_cents
                        trade_reason[trade_count] = 4
                        trade_entry_regime[trade_count] = active_trade_regime
                        trade_count += 1
                        balance += p_cents
                        legs[l_id, 0] = 0.0
                eq[eq_count] = balance
                eq_count += 1
                leg_a_profit_hit = 0
                leg_b_profit_hit = 0
                break

            # 5. Dynamic Time Stop (Clears all active legs)
            if leg_idx == 1 and enable_time_stop and time_stop_minutes > 0.0:
                if (time_minutes[i] - time_minutes[leg_entry_i]) >= time_stop_minutes:
                    for l_id in range(3):
                        if legs[l_id, 0] == 1.0:
                            m_pips = ((bar_close - legs[l_id, 3]) / pip_size_price) * int(legs[l_id, 2])
                            p_cents = m_pips * (legs[l_id, 1] * pip_value_per_1lot) - commission_cents
                            trade_entry_idx[trade_count] = int(legs[l_id, 6])
                            trade_exit_idx[trade_count] = i
                            trade_leg[trade_count] = l_id
                            trade_side[trade_count] = int(legs[l_id, 2])
                            trade_entry_px[trade_count] = legs[l_id, 3]
                            trade_exit_px[trade_count] = bar_close
                            trade_move_pips[trade_count] = m_pips
                            trade_pnl_cents[trade_count] = p_cents
                            trade_reason[trade_count] = 5
                            trade_entry_regime[trade_count] = active_trade_regime
                            trade_count += 1
                            balance += p_cents
                            legs[l_id, 0] = 0.0
                    eq[eq_count] = balance
                    eq_count += 1
                    leg_a_profit_hit = 0
                    leg_b_profit_hit = 0
                    break

    # EOD Cleanup
    if legs[0, 0] == 1.0 or legs[1, 0] == 1.0 or legs[2, 0] == 1.0:
        last_i = n - 1
        for leg_idx in range(3):
            if legs[leg_idx, 0] == 1.0:
                m_pips = ((close[last_i] - legs[leg_idx, 3]) / pip_size_price) * int(legs[leg_idx, 2])
                p_cents = m_pips * (legs[leg_idx, 1] * pip_value_per_1lot) - commission_cents
                trade_entry_idx[trade_count] = int(legs[leg_idx, 6])
                trade_exit_idx[trade_count] = last_i
                trade_leg[trade_count] = leg_idx
                trade_side[trade_count] = int(legs[leg_idx, 2])
                trade_entry_px[trade_count] = legs[leg_idx, 3]
                trade_exit_px[trade_count] = close[last_i]
                trade_move_pips[trade_count] = m_pips
                trade_pnl_cents[trade_count] = p_cents
                trade_reason[trade_count] = 6
                trade_entry_regime[trade_count] = active_trade_regime
                trade_count += 1
                balance += p_cents
        eq[eq_count] = balance
        eq_count += 1

    return (
        trade_entry_idx, trade_exit_idx, trade_leg, trade_side, trade_entry_px,
        trade_exit_px, trade_move_pips, trade_pnl_cents, trade_reason,
        trade_entry_regime, trade_count, eq, eq_count, balance
    )

def _safe_float(value, default):
    return float(default) if value is None else float(value)

def run_backtest(timeframe: str, df: pd.DataFrame, signals: pd.Series, entry_params: dict, exit_model: str, exit_params: dict):
    exit_model_map = {
        "fixed_tp": 0, "mr_exit": 1, "fixed_tp_plus_mr": 2, 
        "partial_tp_plus_mr": 3, "partial_tp_mr_time_stop": 4
    }
    exit_mode_code = int(exit_model_map[exit_model])
    leg_a_atr_target = _safe_float(exit_params.get("leg_a_atr_target", -1.0), -1.0) if exit_params else -1.0
    time_stop_minutes = _safe_float(exit_params.get("time_stop_minutes", -1.0), -1.0) if exit_params else -1.0
    trail_mult = _safe_float(exit_params.get("trail_mult", 0.0), 0.0) if exit_params else 0.0

    idx = df.index
    time_minutes = (idx.view("int64") // 60000000000).astype(np.int64)

    sig = signals.reindex(idx).fillna(0).astype(np.int8).to_numpy()
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    close = df["close"].to_numpy(dtype=np.float64)
    spread = df["spread"].fillna(SPREAD_CAP_POINTS).to_numpy(dtype=np.float64)
    atr14 = df["atr14"].to_numpy(dtype=np.float64)
    regime_code = df["regime_code"].fillna(3).to_numpy(dtype=np.int32)
    
    is_m5 = 1 if timeframe == "M5" else 0

    (
        trade_entry_idx, trade_exit_idx, trade_leg, trade_side, trade_entry_px,
        trade_exit_px, trade_move_pips, trade_pnl_cents, trade_reason,
        trade_entry_regime, trade_count, eq, eq_count, ending_balance
    ) = _run_backtest_numba(
        sig, high, low, close, spread, atr14, regime_code, time_minutes,
        float(entry_params["atr_stop"]), float(entry_params["atr_target"]),
        leg_a_atr_target, exit_mode_code, time_stop_minutes, trail_mult,
        float(INITIAL_BALANCE_CENTS), float(PIP_SIZE_PRICE), float(PIP_VALUE_CENTS_PER_1LOT),
        float(SLIPPAGE_PIPS), float(SPREAD_CAP_POINTS), float(COMMISSION_CENTS_PER_TRADE),
        int(is_m5)
    )

    if trade_count == 0:
        return pd.DataFrame(), compute_metrics(pd.DataFrame(), [], INITIAL_BALANCE_CENTS, float(ending_balance))

    reasons = {1: "STOP", 2: "PROFIT_TARGET", 3: "FIXED_TP", 4: "MR_EXIT", 5: "TIME_STOP", 6: "EOD_CLOSE", 7: "REGIME_SHIFT"}
    regime_map = {1: "TREND", 2: "SHOCK", 3: "MR"}

    trades_df = pd.DataFrame(
        {
            "entry_time": idx[trade_entry_idx[:trade_count]],
            "exit_time": idx[trade_exit_idx[:trade_count]],
            "leg": np.where(trade_leg[:trade_count] == 0, "A", np.where(trade_leg[:trade_count] == 1, "B", "C_SCALE")),
            "side": np.where(trade_side[:trade_count] == 1, "BUY", "SELL"),
            "entry_price": trade_entry_px[:trade_count],
            "exit_price": trade_exit_px[:trade_count],
            "move_pips": trade_move_pips[:trade_count],
            "pnl_cents": trade_pnl_cents[:trade_count],
            "exit_reason": [reasons[int(x)] for x in trade_reason[:trade_count]],
            "entry_regime": [regime_map[int(x)] for x in trade_entry_regime[:trade_count]],
            "exit_model": exit_model,
        }
    )

    return trades_df, compute_metrics(trades_df, eq[:eq_count].tolist(), INITIAL_BALANCE_CENTS, float(ending_balance))

In [8]:
# Vectorized Plateau Stability Engine & Group Experiment Runner

import zlib

def _is_numeric_grid(values: list) -> bool:
    return all(isinstance(v, (int, float, np.integer, np.floating)) for v in values)

def build_step_map(param_grid: dict) -> dict:
    step_map = {}
    for key, vals in param_grid.items():
        uniq = sorted(set(vals), key=lambda x: str(x))
        if _is_numeric_grid(uniq) and len(uniq) > 1:
            diffs = np.diff(np.array(uniq, dtype=float))
            diffs = diffs[diffs > 0]
            step_map[key] = float(np.min(diffs)) if len(diffs) > 0 else 1.0
        else:
            step_map[key] = 1.0 if _is_numeric_grid(uniq) else None
    return step_map

def add_parameter_stability_score(
    results_df: pd.DataFrame,
    param_cols: list[str],
    step_map: dict,
    perf_col: str = "profit_per_trade",
    block_size: int = 128,  # lower to reduce memory pressure
) -> pd.DataFrame:
    if results_df.empty:
        out = results_df.copy()
        out["parameter_stability_score"] = np.nan
        return out

    out = results_df.copy()
    M = len(out)

    numeric_cols = [c for c in param_cols if step_map.get(c, None) is not None]
    cat_cols = [c for c in param_cols if step_map.get(c, None) is None]

    num_mat = out[numeric_cols].astype(float).to_numpy() if numeric_cols else np.empty((M, 0), dtype=float)
    step_sizes = np.array([float(step_map[c]) for c in numeric_cols], dtype=float) if numeric_cols else np.empty((0,), dtype=float)
    cat_mat = np.column_stack([pd.factorize(out[c].astype(str))[0] for c in cat_cols]).astype(np.int32) if cat_cols else np.empty((M, 0), dtype=np.int32)

    perf = out[perf_col].astype(float).to_numpy()
    perf2 = perf * perf
    scores = np.empty(M, dtype=float)

    for start in range(0, M, block_size):
        end = min(M, start + block_size)

        if num_mat.shape[1] > 0:
            diffs = np.abs(num_mat[start:end, None, :] - num_mat[None, :, :])
            num_mask = np.all(diffs <= step_sizes, axis=2)
        else:
            num_mask = np.ones((end - start, M), dtype=bool)

        if cat_mat.shape[1] > 0:
            cat_mask = np.all(cat_mat[start:end, None, :] == cat_mat[None, :, :], axis=2)
        else:
            cat_mask = np.ones((end - start, M), dtype=bool)

        mask = num_mask & cat_mask
        counts = mask.sum(axis=1).astype(np.float64)
        sums = mask @ perf
        sums2 = mask @ perf2

        means = np.where(counts > 0, sums / counts, -1e9)
        vars_ = np.maximum(np.where(counts > 0, sums2 / counts - means * means, 0.0), 0.0)
        scores[start:end] = means - np.sqrt(vars_)

    out["parameter_stability_score"] = scores
    return out

def _cap_entry_combos(
    entry_combos: list[dict],
    timeframe: str,
    strategy_name: str,
    exit_model: str,
) -> tuple[list[dict], int, bool]:
    enable = bool(globals().get("ENABLE_ENTRY_CAP", False))
    cap_map = globals().get("ENTRY_CAP_BY_TF", {})
    cap_val = cap_map.get(timeframe, None) if isinstance(cap_map, dict) else None

    original_n = len(entry_combos)
    if (not enable) or (cap_val is None):
        return entry_combos, original_n, False

    cap_n = int(cap_val)
    if cap_n <= 0 or original_n <= cap_n:
        return entry_combos, original_n, False

    seed_base = int(globals().get("ENTRY_CAP_SEED_BASE", 42))
    seed_key = f"{seed_base}|{timeframe}|{strategy_name}|{exit_model}"
    seed = zlib.crc32(seed_key.encode("utf-8")) & 0xFFFFFFFF

    rng = np.random.default_rng(seed)
    picked_idx = np.sort(rng.choice(original_n, size=cap_n, replace=False))
    capped = [entry_combos[i] for i in picked_idx.tolist()]
    return capped, original_n, True

def get_exit_grid_for_mode(timeframe: str, exit_model: str):
    ts_grid = TIME_STOP_GRID_BY_TF[timeframe]
    tr_grid = TRAIL_MULT_GRID

    if exit_model == "fixed_tp":
        cfg = [{"leg_a_atr_target": p} for p in LEG_A_ATR_TARGET_GRID]
        grid_map = {"leg_a_atr_target": LEG_A_ATR_TARGET_GRID}
    elif exit_model == "mr_exit":
        cfg = [{"leg_a_atr_target": None}]
        grid_map = {}
    elif exit_model == "fixed_tp_plus_mr":
        cfg = [{"leg_a_atr_target": p} for p in LEG_A_ATR_TARGET_GRID]
        grid_map = {"leg_a_atr_target": LEG_A_ATR_TARGET_GRID}
    elif exit_model == "partial_tp_plus_mr":
        cfg = [{"leg_a_atr_target": p} for p in LEG_A_ATR_TARGET_GRID]
        grid_map = {"leg_a_atr_target": LEG_A_ATR_TARGET_GRID}
    elif exit_model == "partial_tp_mr_time_stop":
        cfg = [
            {
                "leg_a_atr_target": p,
                "time_stop_minutes": t,
                "trail_mult": m,
            }
            for p, t, m in itertools.product(
                LEG_A_ATR_TARGET_GRID,
                ts_grid,
                tr_grid,
            )
        ]
        grid_map = {
            "leg_a_atr_target": LEG_A_ATR_TARGET_GRID,
            "time_stop_minutes": ts_grid,
            "trail_mult": tr_grid,
        }
    else:
        raise ValueError(f"Unsupported exit model: {exit_model}")

    return cfg, grid_map

def run_group(timeframe: str, strategy_name: str, exit_model: str) -> pd.DataFrame:
    df = FEATURES_BY_TF[timeframe]
    strategy = STRATEGIES[strategy_name]

    # Per-TF grid override: rebind the per-strategy param_grid and module-level exit grids
    # for M5 only, so M15 runs are byte-identical to before this change.
    _restore = False
    _orig_grid = dict(strategy.param_grid)
    _orig_legA = list(LEG_A_ATR_TARGET_GRID)
    _orig_entry = list(ENTRY_ATR_TARGET_GRID)
    _orig_atrT = list(ATR_TARGET_GRID)
    if timeframe == "M5":
        if strategy_name == "trend_pullback":
            strategy.param_grid = {**_orig_grid,
                "adx_threshold":    M5_ADX_GRID,
                "pullback_rsi":     M5_PULLBACK_RSI_GRID,
                "confirmation_bars":M5_CONFIRMATION_GRID,
                "atr_stop":         M5_ATR_STOP_GRID,
                "atr_target":       M5_ENTRY_TARGET_GRID,
                "session_filter":   _orig_grid.get("session_filter", SESSION_FILTER_VALUES),
            }
        elif strategy_name == "volatility_expansion":
            strategy.param_grid = {**_orig_grid,
                "atr_stop":         M5_ATR_STOP_GRID,
                "atr_target":       M5_ENTRY_TARGET_GRID,
                "session_filter":   _orig_grid.get("session_filter", SESSION_FILTER_VALUES),
            }
        LEG_A_ATR_TARGET_GRID[:] = M5_LEG_A_TARGET_GRID
        ENTRY_ATR_TARGET_GRID[:] = M5_ENTRY_TARGET_GRID
        ATR_TARGET_GRID[:]       = M5_LEG_A_TARGET_GRID
        _restore = True

    t0 = time.time()
    try:
        entry_combos_full = list(strategy.iter_param_dicts())
        entry_combos, full_n, was_capped = _cap_entry_combos(
            entry_combos_full, timeframe, strategy_name, exit_model
        )

        exit_cfgs, exit_grid_map = get_exit_grid_for_mode(timeframe, exit_model)

        rows = []
        total = len(entry_combos) * len(exit_cfgs)
        done = 0

        if was_capped:
            print(
                f"[{timeframe}][{strategy_name}][{exit_model}] entry combos capped: "
                f"{len(entry_combos)}/{full_n}"
            )
        else:
            print(
                f"[{timeframe}][{strategy_name}][{exit_model}] entry combos used: "
                f"{len(entry_combos)}/{full_n}"
            )

        for i, entry_params in enumerate(entry_combos, 1):
            signals = generate_routed_signals(df, entry_params, strategy_name)

            for exit_cfg in exit_cfgs:
                _, metrics = run_backtest(
                    timeframe=timeframe,
                    df=df,
                    signals=signals,
                    entry_params=entry_params,
                    exit_model=exit_model,
                    exit_params=exit_cfg,
                )

                combined_params = {
                    **entry_params,
                    **{k: v for k, v in exit_cfg.items() if v is not None},
                }

                row = {
                    "timeframe": timeframe,
                    "strategy_name": strategy_name,
                    "exit_model": exit_model,
                    "parameter_set": json.dumps(combined_params, sort_keys=True),
                    **metrics,
                }
                for k, v in combined_params.items():
                    row[k] = v
                rows.append(row)

                done += 1

            if i % 50 == 0 or i == len(entry_combos):
                print(f"[{timeframe}][{strategy_name}][{exit_model}] {done}/{total}")

        res = pd.DataFrame(rows)

        param_cols = list(strategy.param_cols) + list(exit_grid_map.keys())
        full_grid_map = {**strategy.param_grid, **exit_grid_map}
        step_map = build_step_map(full_grid_map)

        res = add_parameter_stability_score(
            res, param_cols=param_cols, step_map=step_map, perf_col="profit_per_trade"
        )

        res["robust_score"] = (
            0.45 * res["parameter_stability_score"].astype(float)
            + 0.35 * res["profit_per_trade"].astype(float)
            + 0.20 * res["profit_factor"].astype(float)
        )

        res = res.sort_values(
            ["robust_score", "profit_per_trade", "profit_factor", "sharpe"],
            ascending=False,
        ).reset_index(drop=True)

        return res
    finally:
        if _restore:
            strategy.param_grid = _orig_grid
            LEG_A_ATR_TARGET_GRID[:] = _orig_legA
            ENTRY_ATR_TARGET_GRID[:] = _orig_entry
            ATR_TARGET_GRID[:]       = _orig_atrT
        elapsed = time.time() - t0
        print(f"[{timeframe}][{strategy_name}][{exit_model}] done in {elapsed/60:.2f} minutes")


In [9]:
# Concurrent execution across timeframe + strategy + exit model

group_tasks = [
    (tf, strategy_name, exit_model)
    for tf in TIMEFRAMES
    for strategy_name in STRATEGIES.keys()
    for exit_model in EXIT_MODELS
]

print("Total groups:", len(group_tasks))
for t in group_tasks:
    print("Task:", t)

def _run_task(task):
    tf, strategy_name, exit_model = task
    return run_group(tf, strategy_name, exit_model)

if JOBLIB_OK:
    try:
        nj = min(max(1, N_JOBS), len(group_tasks))
        print(f"Running parallel with joblib loky backend... n_jobs={nj}")
        group_results = Parallel(n_jobs=nj, backend="loky", verbose=10, batch_size=1)(
            delayed(_run_task)(task) for task in group_tasks
        )
    except Exception as e:
        print(f"[warn] loky failed: {type(e).__name__}: {e}")
        try:
            nj = min(max(1, N_JOBS), len(group_tasks))
            print(f"Falling back to threading... n_jobs={nj}")
            group_results = Parallel(n_jobs=nj, backend="threading", verbose=10, batch_size=1)(
                delayed(_run_task)(task) for task in group_tasks
            )
        except Exception as e2:
            print(f"[warn] threading failed: {type(e2).__name__}: {e2}")
            print("Falling back to sequential execution.")
            group_results = [_run_task(task) for task in group_tasks]
else:
    print("joblib unavailable. Running sequentially.")
    group_results = [_run_task(task) for task in group_tasks]

all_results = pd.concat(group_results, axis=0, ignore_index=True)

# Apply statistical significance filter before sorting
valid_results = all_results[all_results["trade_count"] >= 200]

best_rows = (
    valid_results.sort_values(
        ["robust_score", "profit_per_trade", "profit_factor", "sharpe"],
        ascending=False,
    )
    .groupby(["timeframe", "strategy_name", "exit_model"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("All results shape:", all_results.shape)
print("Best rows shape:", best_rows.shape)
display(best_rows.head(20))

Total groups: 20
Task: ('M15', 'trend_pullback', 'fixed_tp')
Task: ('M15', 'trend_pullback', 'mr_exit')
Task: ('M15', 'trend_pullback', 'fixed_tp_plus_mr')
Task: ('M15', 'trend_pullback', 'partial_tp_plus_mr')
Task: ('M15', 'trend_pullback', 'partial_tp_mr_time_stop')
Task: ('M15', 'volatility_expansion', 'fixed_tp')
Task: ('M15', 'volatility_expansion', 'mr_exit')
Task: ('M15', 'volatility_expansion', 'fixed_tp_plus_mr')
Task: ('M15', 'volatility_expansion', 'partial_tp_plus_mr')
Task: ('M15', 'volatility_expansion', 'partial_tp_mr_time_stop')
Task: ('M5', 'trend_pullback', 'fixed_tp')
Task: ('M5', 'trend_pullback', 'mr_exit')
Task: ('M5', 'trend_pullback', 'fixed_tp_plus_mr')
Task: ('M5', 'trend_pullback', 'partial_tp_plus_mr')
Task: ('M5', 'trend_pullback', 'partial_tp_mr_time_stop')
Task: ('M5', 'volatility_expansion', 'fixed_tp')
Task: ('M5', 'volatility_expansion', 'mr_exit')
Task: ('M5', 'volatility_expansion', 'fixed_tp_plus_mr')
Task: ('M5', 'volatility_expansion', 'partial_tp

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:  2.1min
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:  8.1min
[Parallel(n_jobs=6)]: Done  12 out of  20 | elapsed: 11.2min remaining:  7.5min
[Parallel(n_jobs=6)]: Done  15 out of  20 | elapsed: 13.1min remaining:  4.3min
[Parallel(n_jobs=6)]: Done  18 out of  20 | elapsed: 18.7min remaining:  2.1min
[Parallel(n_jobs=6)]: Done  20 out of  20 | elapsed: 49.3min finished


All results shape: (180512, 29)
Best rows shape: (20, 29)


,timeframe,strategy_name,exit_model,parameter_set,profit_factor,sharpe,sortino,calmar,max_drawdown,expectancy,...,atr_target,session_filter,leg_a_atr_target,parameter_stability_score,robust_score,time_stop_minutes,trail_mult,atr_expansion_threshold,breakout_lookback,breakout_buffer
0,M5,trend_pullback,partial_tp_mr_time_stop,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",3.071639,0.870820,43.979937,9.305663,933.225811,87.016885,...,1.0,NY,0.8,38.265071,48.289520,60.0,2.5,NaN,NaN,NaN
1,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 18.0, ""atr_stop"": 3.0, ""atr_...",1.436216,7.868316,11.605853,192.256346,66.635058,45.461365,...,2.0,NY,1.5,25.794567,27.806276,NaN,NaN,NaN,NaN,NaN
2,M15,trend_pullback,fixed_tp_plus_mr,"{""adx_threshold"": 15.0, ""atr_stop"": 3.0, ""atr_...",1.512957,9.715152,12.105875,237.009639,59.858323,38.875593,...,2.0,NY,1.0,25.433380,25.354070,NaN,NaN,NaN,NaN,NaN
3,M5,trend_pullback,fixed_tp,"{""adx_threshold"": 20, ""atr_stop"": 3.0, ""atr_ta...",1.459441,13.455343,11.819368,270.400128,69.580931,23.034638,...,1.0,NY,1.0,11.224739,13.405144,NaN,NaN,NaN,NaN,NaN
4,M5,trend_pullback,fixed_tp_plus_mr,"{""adx_threshold"": 20, ""atr_stop"": 3.0, ""atr_ta...",1.454350,13.275749,11.653971,265.523484,69.580931,22.619211,...,1.0,NY,1.0,11.173342,13.235598,NaN,NaN,NaN,NaN,NaN
5,M15,trend_pullback,mr_exit,"{""adx_threshold"": 18.0, ""atr_stop"": 3.0, ""atr_...",1.138342,1.984898,6.930217,58.674441,166.776124,23.789698,...,2.0,NY,NaN,9.248530,12.715901,NaN,NaN,NaN,NaN,NaN
6,M15,trend_pullback,partial_tp_plus_mr,"{""adx_threshold"": 15.0, ""atr_stop"": 3.0, ""atr_...",1.142225,1.780088,4.255457,26.722583,166.361699,14.291516,...,2.5,NY,1.0,11.242208,10.289469,NaN,NaN,NaN,NaN,NaN
7,M5,trend_pullback,mr_exit,"{""adx_threshold"": 20, ""atr_stop"": 3.0, ""atr_ta...",1.118815,1.652703,8.345260,69.579264,115.796567,16.492303,...,1.0,London,NaN,5.680288,8.552199,NaN,NaN,NaN,NaN,NaN
8,M5,trend_pullback,partial_tp_plus_mr,"{""adx_threshold"": 25, ""atr_stop"": 1.0, ""atr_ta...",1.861395,0.681199,83.373257,2.145046,3169.198704,25.563084,...,1.0,NY,0.8,-3.391031,7.793394,NaN,NaN,NaN,NaN,NaN
9,M15,trend_pullback,partial_tp_mr_time_stop,"{""adx_threshold"": 15.0, ""atr_stop"": 3.0, ""atr_...",1.117067,1.828302,2.764811,20.302758,134.153603,11.389830,...,2.5,NY,1.0,3.620034,5.838869,180.0,2.5,NaN,NaN,NaN


In [10]:
# Reporting and leaderboards (use valid_results to exclude low-trade runs)

os.makedirs("reports", exist_ok=True)
leg_c_lot_rule = "A_if_A_hits_first_else_B"
# Required strategy summary (plus useful extras) -- filtered
summary_cols = [
    "timeframe",
    "strategy_name",
    "exit_model",
    "profit_factor",
    "sharpe",
    "sortino",
    "calmar",
    "max_drawdown",
    "expectancy",
    "win_rate",
    "trade_count",
    "parameter_set",
    "profit_per_trade",
    "parameter_stability_score",
    "robust_score",
]
strategy_summary = valid_results[summary_cols].copy()
strategy_summary["leg_c_lot_rule"] = leg_c_lot_rule
strategy_summary.to_csv("reports/strategy_summary.csv", index=False)

# Full output for diagnostics (unfiltered)
all_results.to_csv("reports/strategy_full_results.csv", index=False)

# New required leaderboard format -- filtered
leaderboard = (
    valid_results.sort_values(
        ["profit_per_trade", "robust_score", "profit_factor", "sharpe"],
        ascending=False,
    )
    .copy()
)
leaderboard = leaderboard.assign(leg_c_lot_rule=leg_c_lot_rule)
leaderboard_view = leaderboard[
    [
        "timeframe",
        "strategy_name",
        "exit_model",
        "profit_factor",
        "sharpe",
        "sortino",
        "calmar",
        "expectancy",
        "profit_per_trade",
        "trade_count",
        "max_drawdown",
        "parameter_set",
        "parameter_stability_score",
        "robust_score",
    ]
].copy()

leaderboard_view = leaderboard_view.rename(
    columns={
        "timeframe": "Timeframe",
        "strategy_name": "Strategy",
        "exit_model": "Exit Model",
        "profit_factor": "PF",
        "sharpe": "Sharpe",
        "sortino": "Sortino",
        "calmar": "Calmar",
        "expectancy": "Expectancy",
        "profit_per_trade": "Profit Per Trade",
        "trade_count": "Trade Count",
        "max_drawdown": "Max DD",
        "leg_c_lot_rule": "Leg C Lot Rule",
    },
)

leaderboard_view.to_csv("reports/strategy_leaderboard.csv", index=False)

# Focus comparison requested in objective -- filtered
focus = valid_results[
    (valid_results["strategy_name"].isin(["trend_pullback", "volatility_expansion"]))
    & (
        valid_results["exit_model"].isin(
            ["partial_tp_plus_mr", "partial_tp_mr_time_stop"]
        )
    )
].copy()

focus = focus.sort_values(
    ["timeframe", "profit_per_trade", "robust_score", "profit_factor", "sharpe"],
    ascending=[True, False, False, False, False],
).reset_index(drop=True)
focus["leg_c_lot_rule"] = leg_c_lot_rule
focus.to_csv("reports/strategy_focus_partialtp_mr.csv", index=False)

print("Saved: reports/strategy_summary.csv")
print("Saved: reports/strategy_full_results.csv")
print("Saved: reports/strategy_leaderboard.csv")
print("Saved: reports/strategy_focus_partialtp_mr.csv")

print("\nTop Leaderboard Rows")
display(leaderboard_view.head(30))

print("\nTop Focus Rows (partial TP + MR variants)")
display(
    focus[
        [
            "timeframe",
            "strategy_name",
            "exit_model",
            "profit_factor",
            "sharpe",
            "sortino",
            "calmar",
            "expectancy",
            "profit_per_trade",
            "trade_count",
            "max_drawdown",
            "parameter_set",
            "parameter_stability_score",
            "robust_score",
            "leg_c_lot_rule",
        ]
    ].head(30)
)

Saved: reports/strategy_summary.csv
Saved: reports/strategy_full_results.csv
Saved: reports/strategy_leaderboard.csv
Saved: reports/strategy_focus_partialtp_mr.csv

Top Leaderboard Rows


,Timeframe,Strategy,Exit Model,PF,Sharpe,Sortino,Calmar,Expectancy,Profit Per Trade,Trade Count,Max DD,parameter_set,parameter_stability_score,robust_score
156752,M5,trend_pullback,partial_tp_mr_time_stop,3.071639,0.870820,43.979937,9.305663,87.016885,87.016885,1497,933.225811,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",38.265071,48.289520
156753,M5,trend_pullback,partial_tp_mr_time_stop,3.071639,0.870820,43.979937,9.305663,87.016885,87.016885,1497,933.225811,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",38.265071,48.289520
156754,M5,trend_pullback,partial_tp_mr_time_stop,3.071639,0.870820,43.979937,9.305663,87.016885,87.016885,1497,933.225811,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",38.265071,48.289520
156755,M5,trend_pullback,partial_tp_mr_time_stop,3.071639,0.870820,43.979937,9.305663,87.016885,87.016885,1497,933.225811,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",38.265071,48.289520
156756,M5,trend_pullback,partial_tp_mr_time_stop,3.071639,0.870820,43.979937,9.305663,87.016885,87.016885,1497,933.225811,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",38.265071,48.289520
156757,M5,trend_pullback,partial_tp_mr_time_stop,3.071639,0.870820,43.979937,9.305663,87.016885,87.016885,1497,933.225811,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",38.265071,48.289520
156758,M5,trend_pullback,partial_tp_mr_time_stop,3.104655,0.880086,45.490094,9.784831,86.327766,86.327766,1525,896.965545,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",38.265071,48.054931
156759,M5,trend_pullback,partial_tp_mr_time_stop,3.104655,0.880086,45.490094,9.784831,86.327766,86.327766,1525,896.965545,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",38.265071,48.054931
156760,M5,trend_pullback,partial_tp_mr_time_stop,3.104655,0.880086,45.490094,9.784831,86.327766,86.327766,1525,896.965545,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",38.265071,48.054931
156761,M5,trend_pullback,partial_tp_mr_time_stop,3.104655,0.880086,45.490094,9.784831,86.327766,86.327766,1525,896.965545,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",38.265071,48.054931



Top Focus Rows (partial TP + MR variants)


,timeframe,strategy_name,exit_model,profit_factor,sharpe,sortino,calmar,expectancy,profit_per_trade,trade_count,max_drawdown,parameter_set,parameter_stability_score,robust_score,leg_c_lot_rule
0,M15,trend_pullback,partial_tp_plus_mr,1.187376,1.566144,4.502738,14.130484,20.688741,20.688741,3242,316.445391,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",6.081038,10.215002,A_if_A_hits_first_else_B
1,M15,trend_pullback,partial_tp_plus_mr,1.187376,1.566144,4.502738,14.130484,20.688741,20.688741,3242,316.445391,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",6.081038,10.215002,A_if_A_hits_first_else_B
2,M15,trend_pullback,partial_tp_plus_mr,1.187376,1.566144,4.502738,14.130484,20.688741,20.688741,3242,316.445391,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",6.081038,10.215002,A_if_A_hits_first_else_B
3,M15,trend_pullback,partial_tp_plus_mr,1.153736,1.386616,3.978869,11.903603,20.220514,20.220514,2920,330.678030,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",2.618387,8.486201,A_if_A_hits_first_else_B
4,M15,trend_pullback,partial_tp_plus_mr,1.153736,1.386616,3.978869,11.903603,20.220514,20.220514,2920,330.678030,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",2.618387,8.486201,A_if_A_hits_first_else_B
5,M15,trend_pullback,partial_tp_plus_mr,1.153736,1.386616,3.978869,11.903603,20.220514,20.220514,2920,330.678030,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",2.618387,8.486201,A_if_A_hits_first_else_B
6,M15,trend_pullback,partial_tp_plus_mr,1.184013,1.618342,4.833370,13.295950,20.138917,20.138917,3499,353.320998,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",6.081038,10.021891,A_if_A_hits_first_else_B
7,M15,trend_pullback,partial_tp_plus_mr,1.184013,1.618342,4.833370,13.295950,20.138917,20.138917,3499,353.320998,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",6.081038,10.021891,A_if_A_hits_first_else_B
8,M15,trend_pullback,partial_tp_plus_mr,1.184013,1.618342,4.833370,13.295950,20.138917,20.138917,3499,353.320998,"{""adx_threshold"": 25.0, ""atr_stop"": 3.0, ""atr_...",6.081038,10.021891,A_if_A_hits_first_else_B
9,M15,trend_pullback,partial_tp_mr_time_stop,1.305670,0.806790,15.614715,7.278333,19.200557,19.200557,4292,754.832118,"{""adx_threshold"": 18.0, ""atr_stop"": 2.5, ""atr_...",-5.502251,4.505316,A_if_A_hits_first_else_B


In [17]:
# Export top robust candidates for downstream wiring into GoldRegimeX_Explorer
# FIX (2026-07-07): previously exported only the top 30 rows by profit_per_trade,
# which meant strategies that survived the 25% Max DD filter but had lower
# profit-per-trade were dropped from the CSV, causing the Explorer to raise
# "No baseline survived 25%% DD filter" even when survivors clearly existed
# (see Cell 12 output). Now the CSV always includes every DD-surviving row
# from valid_results, plus the top N by profit_per_trade for downstream
# ranking flexibility.

leg_c_lot_rule = "A_if_A_hits_first_else_B"
HANDOFF_MAX_DD_PCT = 30.0

handoff_cols = [
    "timeframe",
    "strategy_name",
    "exit_model",
    "parameter_set",
    "profit_factor",
    "sharpe",
    "sortino",
    "calmar",
    "max_drawdown",
    "expectancy",
    "win_rate",
    "trade_count",
    "profit_per_trade",
    "parameter_stability_score",
    "robust_score",
    "leg_c_lot_rule",
]

top_n = 30

# 1) top-N by profit_per_trade / robust_score (previous behaviour)
top_by_profit = (
    valid_results.sort_values(
        ["profit_per_trade", "robust_score", "profit_factor", "sharpe"],
        ascending=False,
    )
    .head(top_n)
    .copy()
)

# 2) ALL rows that survive the 25% Max DD cap - the Explorer needs these.
dd_survivors = valid_results[valid_results["max_drawdown"] <= HANDOFF_MAX_DD_PCT].copy()

# 3) For each timeframe, guarantee at least the top-K DD-survivors by robust_score
#    are present, so both M15 and M5 have baselines to hand off.
per_tf_k = 10
per_tf_survivors = (
    dd_survivors
    .sort_values(["robust_score", "profit_per_trade"], ascending=False)
    .groupby("timeframe", as_index=False)
    .head(per_tf_k)
)

# 4) Union (DD-survivors first so they sort earliest by construction), then dedupe.
handoff = pd.concat([per_tf_survivors, dd_survivors, top_by_profit], ignore_index=True)
handoff = handoff.drop_duplicates(
    subset=["timeframe", "strategy_name", "exit_model", "parameter_set"],
    keep="first",
).reset_index(drop=True)
handoff["leg_c_lot_rule"] = leg_c_lot_rule
handoff = handoff[handoff_cols].copy()

import os
os.makedirs("reports", exist_ok=True)
handoff.to_csv("reports/strategy_winners_for_explorer.csv", index=False)
print("Saved: reports/strategy_winners_for_explorer.csv")
print("  Total rows:", len(handoff))
print("  DD<=%.1f%% rows:" % HANDOFF_MAX_DD_PCT,
      int((handoff["max_drawdown"] <= HANDOFF_MAX_DD_PCT).sum()))
print("  Per timeframe DD survivors:")
print(handoff[handoff["max_drawdown"] <= HANDOFF_MAX_DD_PCT]
      .groupby("timeframe").size().to_string())
display(handoff)


Saved: reports/strategy_winners_for_explorer.csv
  Total rows: 48
  DD<=30.0% rows: 18
  Per timeframe DD survivors:
timeframe
M15     7
M5     11


,timeframe,strategy_name,exit_model,parameter_set,profit_factor,sharpe,sortino,calmar,max_drawdown,expectancy,win_rate,trade_count,profit_per_trade,parameter_stability_score,robust_score,leg_c_lot_rule
0,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 18.0, ""atr_stop"": 2.5, ""atr_...",1.429333,7.588856,10.325806,445.693771,23.832644,34.472288,0.593899,4622,34.472288,25.738594,23.933535,A_if_A_hits_first_else_B
1,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 20.0, ""atr_stop"": 2.5, ""atr_...",1.398660,7.013483,9.296813,336.568216,27.716716,32.123160,0.593664,4356,32.123160,25.738594,23.105205,A_if_A_hits_first_else_B
2,M15,trend_pullback,fixed_tp_plus_mr,"{""adx_threshold"": 18.0, ""atr_stop"": 2.5, ""atr_...",1.409717,7.410074,9.714400,412.697905,23.832644,31.830687,0.593096,4635,31.830687,22.049867,21.345124,A_if_A_hits_first_else_B
3,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 18.0, ""atr_stop"": 2.5, ""atr_...",1.359506,6.271294,9.441870,346.366102,26.964356,30.608058,0.561503,4577,30.608058,21.422354,20.624781,A_if_A_hits_first_else_B
4,M15,trend_pullback,fixed_tp_plus_mr,"{""adx_threshold"": 18.0, ""atr_stop"": 2.5, ""atr_...",1.341413,6.278581,8.816845,299.186613,28.212516,27.566320,0.566950,4593,27.566320,19.305696,18.604058,A_if_A_hits_first_else_B
5,M15,trend_pullback,fixed_tp,"{""adx_threshold"": 20.0, ""atr_stop"": 2.5, ""atr_...",1.247277,4.445328,6.196816,232.176229,27.767681,22.536689,0.537404,4291,22.536689,20.183972,17.220084,A_if_A_hits_first_else_B
6,M15,trend_pullback,fixed_tp_plus_mr,"{""adx_threshold"": 20.0, ""atr_stop"": 2.5, ""atr_...",1.261302,4.696795,6.812928,224.055109,28.526728,22.259899,0.546088,4307,22.259899,18.461599,16.350945,A_if_A_hits_first_else_B
7,M5,trend_pullback,fixed_tp,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",1.586836,15.381944,11.505333,632.912847,29.253526,20.555398,0.765598,13511,20.555398,9.507829,11.790280,A_if_A_hits_first_else_B
8,M5,trend_pullback,fixed_tp_plus_mr,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",1.587033,15.384886,11.508290,654.262362,28.300678,20.556659,0.765524,13511,20.556659,9.481696,11.779001,A_if_A_hits_first_else_B
9,M5,trend_pullback,fixed_tp,"{""adx_threshold"": 25, ""atr_stop"": 3.0, ""atr_ta...",1.588583,15.596669,11.702339,650.504643,29.009494,20.507293,0.764834,13803,20.507293,9.507829,11.773792,A_if_A_hits_first_else_B


In [12]:
# Regime Performance Attribution Analytics Report

def generate_regime_report(trades_df: pd.DataFrame):
    if trades_df.empty:
        print("Regime Report Error: Zero transaction logs generated.")
        return

    regimes = ["TREND", "SHOCK", "MR"]
    print("=" * 50)
    print("      REGIME RESEARCH ATTRIBUTION SUMMARY      ")
    print("=" * 50)

    total_pnl = trades_df["pnl_cents"].sum()

    for regime in regimes:
        r_trades = trades_df[trades_df["entry_regime"] == regime]

        if r_trades.empty:
            print(f"\nREGIME: {regime}\n" + "-" * 20)
            print("Buys: 0 | Sells: 0\nNet Profit: 0.00 Cents\nProfit Factor: N/A\nPnL Contribution: 0.0%")
            continue

        buys = len(r_trades[r_trades["side"] == "BUY"])
        sells = len(r_trades[r_trades["side"] == "SELL"])

        r_pnl = r_trades["pnl_cents"].to_numpy()
        gross_profit = np.sum(r_pnl[r_pnl > 0])
        gross_loss = -np.sum(r_pnl[r_pnl < 0])

        pf = gross_profit / gross_loss if gross_loss > 0 else (10.0 if gross_profit > 0 else 0.0)
        net_profit = np.sum(r_pnl)
        contribution = (net_profit / total_pnl * 100.0) if total_pnl != 0 else 0.0

        print(f"\nREGIME: {regime}\n" + "-" * 20)
        print(f"Buys: {buys} | Sells: {sells}")
        print(f"Net Profit: {net_profit:.2f} Cents")
        print(f"Profit Factor: {pf:.2f}")
        print(f"PnL Contribution: {contribution:.1f}%")

    print("=" * 50)


# Run a regime report on the top valid run for EACH Timeframe and EACH Strategy
if valid_results.empty:
    print("No valid_results available (trade_count >= 200). Run the backtest cell first.")
else:
    for tf in TIMEFRAMES:
        for strat_name in STRATEGIES.keys():
            # Filter for specific timeframe and strategy
            strat_results = valid_results[
                (valid_results["strategy_name"] == strat_name)
                & (valid_results["timeframe"] == tf)
            ]

            if strat_results.empty:
                print(f"\nNo valid runs found for {tf} | {strat_name}.")
                continue

            best_strat_row = strat_results.sort_values(
                ["robust_score", "profit_per_trade", "profit_factor", "sharpe"],
                ascending=False,
            ).iloc[0]

            exit_model = best_strat_row["exit_model"]

            combined_params = json.loads(best_strat_row["parameter_set"])
            entry_params = {
                k: combined_params[k]
                for k in STRATEGIES[strat_name].param_cols
                if k in combined_params
            }

            exit_keys = ["leg_a_atr_target", "time_stop_minutes", "trail_mult"]
            exit_params = {k: combined_params[k] for k in exit_keys if k in combined_params}

            df = FEATURES_BY_TF[tf]
            signals = generate_routed_signals(df, entry_params, strat_name)
            
            # FIX: Passed timeframe down correctly here
            trades_df, _ = run_backtest(
                timeframe=tf,
                df=df,
                signals=signals,
                entry_params=entry_params,
                exit_model=exit_model,
                exit_params=exit_params,
            )

            print(f"\n{'=' * 50}")
            print(f"  BEST RUN: {tf} | {strat_name.upper()} | {exit_model}")
            print(f"{'=' * 50}")
            generate_regime_report(trades_df)


  BEST RUN: M15 | TREND_PULLBACK | fixed_tp
      REGIME RESEARCH ATTRIBUTION SUMMARY      

REGIME: TREND
--------------------
Buys: 2198 | Sells: 2029
Net Profit: 52878.48 Cents
Profit Factor: 1.38
PnL Contribution: 100.0%

REGIME: SHOCK
--------------------
Buys: 0 | Sells: 0
Net Profit: 0.00 Cents
Profit Factor: N/A
PnL Contribution: 0.0%

REGIME: MR
--------------------
Buys: 0 | Sells: 0
Net Profit: 0.00 Cents
Profit Factor: N/A
PnL Contribution: 0.0%

  BEST RUN: M15 | VOLATILITY_EXPANSION | mr_exit
      REGIME RESEARCH ATTRIBUTION SUMMARY      

REGIME: TREND
--------------------
Buys: 0 | Sells: 0
Net Profit: 0.00 Cents
Profit Factor: N/A
PnL Contribution: 0.0%

REGIME: SHOCK
--------------------
Buys: 154 | Sells: 158
Net Profit: 1892.06 Cents
Profit Factor: 1.10
PnL Contribution: 100.0%

REGIME: MR
--------------------
Buys: 0 | Sells: 0
Net Profit: 0.00 Cents
Profit Factor: N/A
PnL Contribution: 0.0%

  BEST RUN: M5 | TREND_PULLBACK | partial_tp_mr_time_stop
      REGIME 

In [18]:
MAX_DD_PCT = 30.0  # adjust cap (percent)

cols = [
    "timeframe", "strategy_name", "exit_model",
    "trade_count", "net_profit", "profit_per_trade", "profit_per_trade_usd",
    "max_drawdown", "robust_score", "profit_factor", "sharpe"
]

results_df = globals().get("all_results", None)
if results_df is None:
    results_df = globals().get("valid_results", None)
if results_df is None:
    raise NameError("all_results and valid_results are not defined. Run the backtest cell first.")

filtered = results_df[
    results_df["max_drawdown"] <= MAX_DD_PCT
].copy()
filtered["profit_per_trade_usd"] = filtered["profit_per_trade"] / 100.0

display(
    filtered[cols]
    .sort_values(["timeframe", "robust_score"], ascending=[True, False])
    .reset_index(drop=True)
)

print("Remaining rows:", len(filtered))

,timeframe,strategy_name,exit_model,trade_count,net_profit,profit_per_trade,profit_per_trade_usd,max_drawdown,robust_score,profit_factor,sharpe
0,M15,trend_pullback,fixed_tp,4622,159330.913451,34.472288,0.344723,23.832644,23.933535,1.429333,7.588856
1,M15,trend_pullback,fixed_tp,4356,139928.484418,32.123160,0.321232,27.716716,23.105205,1.398660,7.013483
2,M15,trend_pullback,fixed_tp_plus_mr,4635,147535.232558,31.830687,0.318307,23.832644,21.345124,1.409717,7.410074
3,M15,trend_pullback,fixed_tp,4577,140093.082774,30.608058,0.306081,26.964356,20.624781,1.359506,6.271294
4,M15,trend_pullback,fixed_tp_plus_mr,4593,126612.105633,27.566320,0.275663,28.212516,18.604058,1.341413,6.278581
5,M15,trend_pullback,fixed_tp,4291,96704.931269,22.536689,0.225367,27.767681,17.220084,1.247277,4.445328
6,M15,trend_pullback,fixed_tp_plus_mr,4307,95873.386126,22.259899,0.222599,28.526728,16.350945,1.261302,4.696795
7,M5,trend_pullback,fixed_tp,13511,277723.984142,20.555398,0.205554,29.253526,11.790280,1.586836,15.381944
8,M5,trend_pullback,fixed_tp_plus_mr,13511,277741.025511,20.556659,0.205567,28.300678,11.779001,1.587033,15.384886
9,M5,trend_pullback,fixed_tp,13803,283062.159678,20.507293,0.205073,29.009494,11.773792,1.588583,15.596669


Remaining rows: 18


In [14]:
# ============================================================================
# Profit-Per-Trade Target Sizing  (position-size solver)
# ----------------------------------------------------------------------------
# WHY per-trade profit looks tiny:
#   The backtest trades a FIXED 0.01 micro-lot (POSITION_A = POSITION_B = 0.01),
#   pip value = 100 cents/lot, commission = 0, so
#       pnl_cents = move_pips * (lot * 100)
#   is PERFECTLY LINEAR in lot size. The ~12.3c (M15) / ~4.0c (M5) figures are
#   therefore per-0.01-lot (i.e. ~$0.12 / ~$0.04 per trade). max_drawdown is a
#   PERCENT of an equity curve seeded by a fixed 1500c ($15) cushion; because
#   accumulated profit dwarfs that cushion, %DD is nearly INVARIANT to lot size.
#   => per-trade profit is essentially a position-size dial you can turn while
#   %DD barely moves.
#
# This cell solves, per candidate, for the position size that reaches a TARGET
# per-trade profit while keeping max_drawdown <= a hard cap. It re-runs only a
# shortlist (no full-grid rerun) and recomputes drawdown at each lot multiplier
# EXACTLY from the trade PnL stream (valid because PnL is linear in lot).
# NOTE: this is POSITION SIZING (leverage), NOT new alpha -- profit factor and
# win rate are unchanged. Mind real-world margin/broker lot limits.
# ============================================================================

TARGET_PPT_USD   = 0.75      # desired profit per trade, in DOLLARS
MAX_DD_CAP_PCT   = 30.0      # hard drawdown cap (percent)
BASE_LOT         = POSITION_A  # current per-leg lot (0.01)
SHORTLIST_PER_TF = 60        # candidates re-run per timeframe (by ppt & robustness)
K_MAX            = 1000.0    # max lot multiplier searched (=> up to BASE_LOT * K_MAX)

TARGET_PPT_CENTS = TARGET_PPT_USD * 100.0


def _dd_pct_for_scale(pnl_cents, k, initial_balance=INITIAL_BALANCE_CENTS):
    """Rebuild equity at lot multiplier k and replicate compute_metrics' drawdown
    (rolling window = min(3, len) mean, peak-to-trough percent)."""
    cum = np.cumsum(pnl_cents) * float(k)
    eq = np.empty(len(cum) + 1, dtype=float)
    eq[0] = initial_balance
    eq[1:] = initial_balance + cum
    window = min(3, len(eq))
    avg_eq = pd.Series(eq).rolling(window=window, min_periods=1).mean().to_numpy()
    peaks = np.maximum.accumulate(avg_eq)
    dd = peaks - avg_eq
    return float(np.max(dd / np.maximum(peaks, 1e-9)) * 100.0) if len(dd) else 0.0


def _max_scale_within_dd(pnl_cents, cap_pct, k_hi=K_MAX):
    """Largest k in (0, k_hi] with max_drawdown <= cap_pct. dd% is monotonically
    increasing in k, so binary-search the crossing."""
    if _dd_pct_for_scale(pnl_cents, k_hi) <= cap_pct:
        return k_hi
    lo, hi = 0.0, k_hi
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if _dd_pct_for_scale(pnl_cents, mid) <= cap_pct:
            lo = mid
        else:
            hi = mid
    return lo


_base = globals().get("valid_results", None)
if _base is None or _base.empty:
    raise NameError("valid_results is not defined/empty. Run the backtest cells first.")

_cand_frames = []
for _tf in TIMEFRAMES:
    _sub = _base[_base["timeframe"] == _tf]
    _by_ppt = _sub.sort_values("profit_per_trade", ascending=False).head(SHORTLIST_PER_TF)
    _by_rob = _sub.sort_values("robust_score", ascending=False).head(SHORTLIST_PER_TF)
    _cand_frames.append(pd.concat([_by_ppt, _by_rob], ignore_index=True))

candidates = (
    pd.concat(_cand_frames, ignore_index=True)
    .drop_duplicates(subset=["timeframe", "strategy_name", "exit_model", "parameter_set"])
    .reset_index(drop=True)
)
print("Sizing shortlist: %d candidates across %d timeframes" % (len(candidates), len(TIMEFRAMES)))

_exit_keys = ["leg_a_atr_target", "time_stop_minutes", "trail_mult"]
rows = []
for _i, r in candidates.iterrows():
    tf = r["timeframe"]
    strat_name = r["strategy_name"]
    exit_model = r["exit_model"]
    params_all = json.loads(r["parameter_set"])
    entry_params = {k: params_all[k] for k in STRATEGIES[strat_name].param_cols if k in params_all}
    exit_params = {k: params_all[k] for k in _exit_keys if k in params_all}

    df = FEATURES_BY_TF[tf]
    signals = generate_routed_signals(df, entry_params, strat_name)
    trades_df, met = run_backtest(
        timeframe=tf, df=df, signals=signals,
        entry_params=entry_params, exit_model=exit_model, exit_params=exit_params,
    )
    if trades_df.empty:
        continue

    pnl = trades_df["pnl_cents"].to_numpy(dtype=float)
    n = int(len(pnl))
    base_ppt_cents = float(np.sum(pnl) / max(n, 1))   # per BASE_LOT
    if base_ppt_cents <= 0.0:
        continue

    dd_base = _dd_pct_for_scale(pnl, 1.0)             # ~ matches met["max_drawdown"]
    k_target = TARGET_PPT_CENTS / base_ppt_cents
    k_maxdd = _max_scale_within_dd(pnl, MAX_DD_CAP_PCT)
    feasible = bool(k_target <= k_maxdd)

    rows.append({
        "timeframe": tf, "strategy_name": strat_name, "exit_model": exit_model,
        "trade_count": n,
        "base_ppt_usd": base_ppt_cents / 100.0,
        "base_max_dd_pct": dd_base,
        "engine_max_dd_pct": float(met.get("max_drawdown", np.nan)),
        "lot_for_target": round(BASE_LOT * k_target, 4),
        "dd_at_target_pct": _dd_pct_for_scale(pnl, k_target),
        "meets_target_within_dd": feasible,
        "max_lot_within_dd": round(BASE_LOT * k_maxdd, 4),
        "max_ppt_usd_within_dd": (base_ppt_cents * k_maxdd) / 100.0,
        "profit_factor": float(r.get("profit_factor", np.nan)),
        "win_rate": float(r.get("win_rate", np.nan)),
        "robust_score": float(r.get("robust_score", np.nan)),
        "parameter_set": r["parameter_set"],
    })

sizing_results = pd.DataFrame(rows)
if sizing_results.empty:
    print("No positive-expectancy candidates to size.")
else:
    os.makedirs("reports", exist_ok=True)
    _sorted = sizing_results.sort_values(
        ["timeframe", "max_ppt_usd_within_dd"], ascending=[True, False]
    ).reset_index(drop=True)
    _sorted.to_csv("reports/profit_per_trade_sizing.csv", index=False)

    print("\nTarget: >= $%.2f / trade   |   Max DD cap: %.1f%%   |   base lot: %.2f\n" %
          (TARGET_PPT_USD, MAX_DD_CAP_PCT, BASE_LOT))

    print("=" * 78)
    print("  RECOMMENDED SIZING PER TIMEFRAME")
    print("=" * 78)
    for tf in TIMEFRAMES:
        sub = sizing_results[sizing_results["timeframe"] == tf]
        if sub.empty:
            print("\n[%s] no candidates." % tf)
            continue
        feas = sub[sub["meets_target_within_dd"]]
        if not feas.empty:
            pick = feas.sort_values("robust_score", ascending=False).iloc[0]
        else:
            pick = sub.sort_values("max_ppt_usd_within_dd", ascending=False).iloc[0]
        print("\n[%s] %s / %s" % (tf, pick["strategy_name"], pick["exit_model"]))
        print("   base profit/trade : $%.4f  at lot %.2f  (DD %.2f%%)" %
              (pick["base_ppt_usd"], BASE_LOT, pick["base_max_dd_pct"]))
        if pick["meets_target_within_dd"]:
            print("   -> set lot = %.4f  =>  profit/trade $%.2f  at DD %.2f%%  (<= %.0f%% cap)" %
                  (pick["lot_for_target"], TARGET_PPT_USD, pick["dd_at_target_pct"], MAX_DD_CAP_PCT))
        else:
            print("   -> target $%.2f NOT reachable within %.0f%% DD." %
                  (TARGET_PPT_USD, MAX_DD_CAP_PCT))
            print("      best within cap: lot %.4f => profit/trade $%.2f at DD ~%.1f%%" %
                  (pick["max_lot_within_dd"], pick["max_ppt_usd_within_dd"], MAX_DD_CAP_PCT))

    print("\nSaved: reports/profit_per_trade_sizing.csv")
    _show = ["timeframe", "strategy_name", "exit_model", "trade_count",
             "base_ppt_usd", "base_max_dd_pct", "lot_for_target", "dd_at_target_pct",
             "meets_target_within_dd", "max_lot_within_dd", "max_ppt_usd_within_dd",
             "profit_factor", "win_rate", "robust_score"]
    display(_sorted[_show])


Sizing shortlist: 156 candidates across 2 timeframes

Target: >= $0.75 / trade   |   Max DD cap: 30.0%   |   base lot: 0.05

  RECOMMENDED SIZING PER TIMEFRAME

[M15] trend_pullback / fixed_tp
   base profit/trade : $0.0910  at lot 0.05  (DD 17.46%)
   -> target $0.75 NOT reachable within 30% DD.
      best within cap: lot 0.1584 => profit/trade $0.29 at DD ~30.0%

[M5] trend_pullback / partial_tp_mr_time_stop
   base profit/trade : $0.1556  at lot 0.05  (DD 361.55%)
   -> target $0.75 NOT reachable within 30% DD.
      best within cap: lot 0.0041 => profit/trade $0.01 at DD ~30.0%

Saved: reports/profit_per_trade_sizing.csv


,timeframe,strategy_name,exit_model,trade_count,base_ppt_usd,base_max_dd_pct,lot_for_target,dd_at_target_pct,meets_target_within_dd,max_lot_within_dd,max_ppt_usd_within_dd,profit_factor,win_rate,robust_score
0,M15,trend_pullback,fixed_tp,4622,0.091004,17.458410,0.4121,75.699049,False,0.1584,0.288388,1.429333,0.593899,23.933535
1,M15,trend_pullback,fixed_tp,4356,0.082892,18.081086,0.4524,79.855182,False,0.1625,0.269397,1.398660,0.593664,23.105205
2,M15,trend_pullback,fixed_tp,4960,0.098132,14.290648,0.3821,86.982607,False,0.1318,0.258672,1.472103,0.597177,24.933152
3,M15,trend_pullback,fixed_tp_plus_mr,4635,0.081327,19.376200,0.4611,84.222189,False,0.1584,0.257722,1.409717,0.593096,21.345124
4,M15,trend_pullback,fixed_tp_plus_mr,4976,0.090183,17.972413,0.4158,94.649220,False,0.1318,0.237720,1.460251,0.596664,22.495731
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,M5,trend_pullback,partial_tp_mr_time_stop,1889,0.121015,434.372700,0.3099,1617.766594,False,0.0030,0.007355,2.632650,0.428269,40.467224
152,M5,trend_pullback,partial_tp_mr_time_stop,1889,0.121015,434.372700,0.3099,1617.766594,False,0.0030,0.007355,2.632650,0.428269,40.467224
153,M5,trend_pullback,partial_tp_mr_time_stop,1889,0.121015,434.372700,0.3099,1617.766594,False,0.0030,0.007355,2.632650,0.428269,40.467224
154,M5,trend_pullback,partial_tp_mr_time_stop,1889,0.121015,434.372700,0.3099,1617.766594,False,0.0030,0.007355,2.632650,0.428269,40.467224
